# DeployTeach Pyannote GPU Evaluation

**Fixed notebook** – resolves all dependency conflicts and runs the pyannote baseline evaluation.

### Prerequisites
- Runtime type: **GPU** (A100 recommended)
- Colab Secret `HF_TOKEN` must be set (you need access to `pyannote/speaker-diarization-3.1`)
- `colab_pyannote_android_pipeline_100.zip` must be uploaded to `/content/`

### Execution order
Run cells **top to bottom, once**. Cell 2 installs packages – after it finishes, **restart the runtime** (Runtime → Restart runtime), then continue from Cell 3.

In [1]:
# ── Cell 1 ── Unzip dataset and change working directory
import os

zip_path = '/content/colab_pyannote_android_pipeline_100.zip'
extract_dir = '/content/deployteach_eval'

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"ZIP not found at {zip_path}.\n"
        "Please upload colab_pyannote_android_pipeline_100.zip to /content/ first."
    )

!unzip -q -o "{zip_path}" -d "{extract_dir}"
%cd /content/deployteach_eval
print("Working directory:", os.getcwd())
print("Contents:", os.listdir('.'))

/content/deployteach_eval
Working directory: /content/deployteach_eval
Contents: ['generated_dataset', 'iot_pyannote.ipynb', 'tools']


In [2]:
# ── Cell 2 ── Install a self-consistent environment
#
# Key decisions:
#   • torch 2.5.1+cu124 / torchaudio 2.5.1+cu124 / torchvision 0.20.1+cu124
#     from the PyTorch CUDA 12.4 index — all three MUST come from the same
#     index to avoid the circular-import torchvision bug.
#   • pyannote.audio 3.3.2 is compatible with torch 2.5.x.
#   • torchmetrics is pinned to <1.6 to avoid the arniqa→torchvision
#     circular-import that caused the previous AttributeError.
#   • No numpy pin needed: pyannote 3.3.2 works fine with numpy 2.x.
#   • Do NOT run apt-get for cuda-toolkit; Colab GPU runtimes already have
#     CUDA libraries in /usr/local/cuda.
#
# After this cell finishes → Runtime → Restart runtime → continue from Cell 3.

!pip install -q \
    torch==2.5.1+cu124 \
    torchaudio==2.5.1+cu124 \
    torchvision==0.20.1+cu124 \
    --extra-index-url https://download.pytorch.org/whl/cu124

!pip install -q \
    "pyannote.audio==3.3.2" \
    "torchmetrics>=0.11.0,<1.6.0"

# Install requirements from the project if present
import os
req = 'tools/pyannote_requirements.txt'
if os.path.exists(req):
    !pip install -q -r "{req}"

print("\n✅ Installation complete. Please restart the runtime now.")
print("   Runtime → Restart runtime  (then continue from Cell 3)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 46.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 129.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 73.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 135.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# ── Cell 3 ── Verify environment (run after runtime restart)
import torch
import torchaudio
import torchvision
import numpy

print(f"torch       : {torch.__version__}")
print(f"torchaudio  : {torchaudio.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"numpy       : {numpy.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device : {torch.cuda.get_device_name(0)}")

import pyannote.audio
print(f"pyannote.audio: {pyannote.audio.__version__}")

torch       : 2.5.1+cu124
torchaudio  : 2.5.1+cu124
torchvision : 0.20.1+cu124
numpy       : 2.4.4
CUDA available: True
CUDA device : NVIDIA A100-SXM4-40GB
pyannote.audio: 3.3.2


In [2]:
# ── Cell 4 ── Change to project directory (needed after runtime restart)
import os

project_dir = '/content/deployteach_eval'
if not os.path.exists(project_dir):
    raise RuntimeError(
        f"{project_dir} not found.\n"
        "Please run Cell 1 again (the unzip step) then restart and resume from Cell 3."
    )
%cd /content/deployteach_eval
print("Working directory:", os.getcwd())

/content/deployteach_eval
Working directory: /content/deployteach_eval


In [3]:
# ── Cell 5 ── Load HF_TOKEN from Colab Secrets
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise ValueError(
        "HF_TOKEN secret is empty or not set.\n"
        "Go to  Secrets (🔑 icon in the left panel) and add your HuggingFace token."
    )
os.environ['HF_TOKEN'] = hf_token
print("✅ HF_TOKEN loaded from Colab Secrets.")

✅ HF_TOKEN loaded from Colab Secrets.


In [4]:
# ── Cell 6 ── Patch pyannote.audio and the baseline script
#
# huggingface_hub ≥0.17 removed 'use_auth_token'; pyannote 3.3.2 still uses
# it internally.  We replace every occurrence with 'token' in the installed
# library files AND in the local baseline script.

import os
import pyannote.audio

# 1. Patch installed pyannote.audio library
lib_dir = os.path.dirname(pyannote.audio.__file__)
patched = 0
for root, dirs, files in os.walk(lib_dir):
    for fname in files:
        if not fname.endswith('.py'):
            continue
        path = os.path.join(root, fname)
        with open(path, 'r', errors='replace') as f:
            content = f.read()
        if 'use_auth_token' in content:
            new_content = content.replace('use_auth_token', 'token')
            with open(path, 'w') as f:
                f.write(new_content)
            patched += 1
print(f"Library patch: {patched} file(s) updated.")

# 2. Patch local baseline script
script_path = 'tools/run_pyannote_baseline.py'
if os.path.exists(script_path):
    with open(script_path, 'r') as f:
        sc = f.read()

    # Replace use_auth_token= references
    sc = sc.replace('use_auth_token=args.hf_token', 'token=args.hf_token')
    sc = sc.replace('use_auth_token=hf_token', 'token=hf_token')

    # Ensure Pipeline.from_pretrained passes the token argument
    if 'token=args.hf_token' not in sc and 'Pipeline.from_pretrained(args.model)' in sc:
        sc = sc.replace(
            'Pipeline.from_pretrained(args.model)',
            'Pipeline.from_pretrained(args.model, token=args.hf_token)'
        )

    with open(script_path, 'w') as f:
        f.write(sc)
    print("Baseline script patched.")
else:
    print(f"WARNING: {script_path} not found – skipping script patch.")

print("\n✅ Patches applied. Ready to run evaluation.")

Library patch: 11 file(s) updated.
Baseline script patched.

✅ Patches applied. Ready to run evaluation.


In [5]:
# ── Cell 7 ── Run the pyannote baseline evaluation
#
# Uses --dataset (single argument) which the script maps to audio/truth/vad
# sub-directories automatically.  Adjust --limit to control how many samples
# are processed (remove the flag to process all).

import os

# Export token so the subprocess can access it via os.environ if needed
token = os.environ.get('HF_TOKEN', '')
os.environ['HF_TOKEN'] = token  # ensure it propagates to child process

!python3 tools/run_pyannote_baseline.py \
  --audio generated_dataset/audio \
  --truth generated_dataset/truth \
  --vad generated_dataset/vad \
  --out generated_dataset/pyannote_android_pipeline \
  --device cuda \
  --hf-token {token} \
  --score

config.yaml: 100% 469/469 [00:00<00:00, 2.14MB/s]
pytorch_model.bin: 100% 5.91M/5.91M [00:01<00:00, 4.61MB/s]
config.yaml: 100% 399/399 [00:00<00:00, 2.18MB/s]
/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of th

In [6]:
!python3 tools/score_dataset.py --truth generated_dataset/truth --pred generated_dataset/pyannote_android_pipeline/predictions --out generated_dataset/pyannote_android_pipeline/metrics.json



Scored files: 100
Accuracy: 0.8635
SILENCE: precision=0.9842 recall=0.9152 f1=0.9484 supportBins=47285
INSTRUCTOR: precision=0.6862 recall=0.5904 f1=0.6347 supportBins=4993
STUDENT: precision=0.4566 recall=0.8217 f1=0.5870 supportBins=6220
BOTH: precision=0.8868 recall=0.3182 f1=0.4684 supportBins=1502
Wrote metrics to generated_dataset/pyannote_android_pipeline/metrics.json.


In [ ]:
# ── Cell 8 (optional) ── Run with a sample limit for quick testing
# Useful for verifying the pipeline works before running the full dataset.

!python3 tools/run_pyannote_baseline.py \
  --dataset generated_dataset \
  --out generated_dataset/pyannote_baseline_10 \
  --limit 10 \
  --device cuda \
  --hf-token "$HF_TOKEN" \
  --score